In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn joblib -q
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import joblib, warnings
warnings.filterwarnings('ignore')
print('✅ Libraries ready!')

✅ Libraries ready!


In [ ]:
np.random.seed(42)
N = 1000
data = []
for i in range(N):
    d = np.random.choice([0,1], p=[0.70,0.30])
    if d == 1:
        hr=np.random.normal(115,15); spo2=np.random.normal(88,4); rr=np.random.normal(26,4)
        sbp=np.random.normal(82,10); temp=np.random.normal(38.8,0.5); gcs=np.random.normal(9,2)
        lactate=np.random.normal(4.2,1.0); wbc=np.random.normal(14.5,3.0)
    else:
        hr=np.random.normal(78,10); spo2=np.random.normal(97,2); rr=np.random.normal(15,3)
        sbp=np.random.normal(118,12); temp=np.random.normal(36.8,0.4); gcs=np.random.normal(14,1)
        lactate=np.random.normal(1.2,0.4); wbc=np.random.normal(7.5,2.0)
    data.append({'hr':round(np.clip(hr,30,200),1),'spo2':round(np.clip(spo2,70,100),1),
        'rr':round(np.clip(rr,5,50),1),'sbp':round(np.clip(sbp,50,200),1),
        'temp':round(np.clip(temp,34,42),2),'gcs':round(np.clip(gcs,3,15)),
        'lactate':round(np.clip(lactate,0.5,15),2),'wbc':round(np.clip(wbc,1,30),2),'label':d})
df = pd.DataFrame(data)
print(f'✅ Dataset ready! {N} patients | Deteriorating: {df.label.sum()} ({df.label.mean()*100:.0f}%)')
df.head()

✅ Dataset ready! 1000 patients | Deteriorating: 270 (27%)


,hr,spo2,rr,sbp,temp,gcs,lactate,wbc,label
0,66.9,97.6,15.8,130.1,36.57,13,0.97,5.65,0
1,60.8,95.9,12.0,121.8,36.44,13,1.79,7.05,0
2,103.6,97.8,15.4,111.8,36.56,15,1.32,6.23,0
3,77.9,94.9,17.5,103.3,36.88,12,0.67,7.89,0
4,118.3,91.5,22.0,66.2,39.19,8,2.85,11.86,1


In [ ]:
X = df[['hr','spo2','rr','sbp','temp','gcs','lactate','wbc']].values
y = df['label'].values
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
model = RandomForestClassifier(n_estimators=200,max_depth=10,class_weight='balanced',random_state=42,n_jobs=-1)
model.fit(X_train,y_train)
cv = cross_val_score(model,X_train,y_train,cv=5,scoring='roc_auc')
print(f'✅ Model trained! CV AUROC: {cv.mean():.4f} ± {cv.std():.4f}')

✅ Model trained! CV AUROC: 1.0000 ± 0.0000


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]
print(f'Accuracy: {accuracy_score(y_test,y_pred)*100:.2f}%')
print(f'AUROC:    {roc_auc_score(y_test,y_prob):.4f}')
print(classification_report(y_test,y_pred,target_names=['Stable','Deteriorating']))

Accuracy: 100.00%
AUROC:    1.0000
               precision    recall  f1-score   support

       Stable       1.00      1.00      1.00       146
Deteriorating       1.00      1.00      1.00        54

     accuracy                           1.00       200
    macro avg       1.00      1.00      1.00       200
 weighted avg       1.00      1.00      1.00       200



In [ ]:
def predict(name,hr,spo2,rr,sbp,temp,gcs,lac,wbc):
    v = scaler.transform([[hr,spo2,rr,sbp,temp,gcs,lac,wbc]])
    score = model.predict_proba(v)[0][1]*100
    level = '🔴 HIGH RISK' if score>=70 else '🟡 MEDIUM' if score>=40 else '🟢 LOW RISK'
    print(f'{name}: Score={score:.1f}% → {level}')

predict('Ramesh Kumar', 118,88,28,78,39.2,11,4.5,15.2)
predict('Sunita Devi',  105,82,26,95,38.1,14,3.8,13.5)
predict('Priya Mehta',   76,97,16,112,37.1,15,1.0,6.5)
predict('Meena Patel',  125,85,30,82,38.7,10,5.2,16.8)

Ramesh Kumar: Score=100.0% → 🔴 HIGH RISK
Sunita Devi: Score=99.5% → 🔴 HIGH RISK
Priya Mehta: Score=0.0% → 🟢 LOW RISK
Meena Patel: Score=100.0% → 🔴 HIGH RISK
